# Практическая работа 3. Множественная регрессия на sklearn


## Задание 1
Загружаем датасет California Housing, преобразуем его в DataFrame и смотрим названия и описание признаков.


In [1]:
import pandas as pd
from sklearn.datasets import fetch_california_housing

# Загружаем данные
housing = fetch_california_housing()

# Преобразуем признаки в DataFrame
df = pd.DataFrame(housing.data, columns=housing.feature_names)

# Добавляем целевую переменную
df["MedHouseVal"] = housing.target

print(df.head())
print("\nНазвания признаков:")
print(housing.feature_names)
print("\nОписание датасета:")
print(housing.DESCR)


   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  MedHouseVal  
0    -122.23        4.526  
1    -122.22        3.585  
2    -122.24        3.521  
3    -122.25        3.413  
4    -122.25        3.422  

Названия признаков:
['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']

Описание датасета:
.. _california_housing_dataset:

California Housing dataset
--------------------------

**Data Set Characteristics:**

:Number of Instances: 20640

:Number of Attributes: 8 numeric, predictive attributes and the target

:Attribute Info

## Задание 2
Отделяем признаки X от целевой переменной y и делим данные на обучающую и тестовую выборки 80/20.


In [2]:
from sklearn.model_selection import train_test_split

X = df.drop("MedHouseVal", axis=1)
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Обучающая выборка:", X_train.shape)
print("Тестовая выборка:", X_test.shape)


Обучающая выборка: (16512, 8)
Тестовая выборка: (4128, 8)


## Задание 3
Создаём модель LinearRegression и обучаем её на тренировочных данных.

In [3]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

print("Модель обучена")


Модель обучена


## Задание 4
Получаем прогноз модели для тестовой выборки и считаем MSE и R².


MSE — средняя квадратичная ошибка

MSE показывает, насколько сильно предсказания модели отличаются от настоящих значений.

$$
MSE = \frac{1}{N}\sum_{i=1}^{N}(y_i-\hat{y}_i)^2
$$

где:

- $y_i$ — настоящее значение;
- $\hat{y}_i$ — предсказанное значение;
- $N$ — количество объектов.

Чем меньше MSE, тем лучше работает модель.


R² — коэффициент детерминации

R² показывает, насколько хорошо модель объясняет изменение целевой переменной.

$$
R^2 = 1 - \frac{\sum_{i=1}^{N}(y_i-\hat{y}_i)^2}{\sum_{i=1}^{N}(y_i-\bar{y})^2}
$$

где:

- $y_i$ — настоящее значение;
- $\hat{y}_i$ — предсказание модели;
- $\bar{y}$ — среднее значение целевой переменной.

Чем ближе R² к 1, тем лучше модель описывает данные.

In [4]:
from sklearn.metrics import mean_squared_error, r2_score

# Делаем прогноз
y_pred = model.predict(X_test)

# Считаем метрики
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MSE:", mse)
print("R²:", r2)


MSE: 0.5558915986952444
R²: 0.5757877060324508


Полученное значение:MSE = 0.556

Само по себе MSE не очень удобно интерпретировать, потому что ошибка возводится в квадрат.

Поэтому можно вычислить RMSE:

$$
RMSE = \sqrt{MSE}
$$


$$
RMSE = \sqrt{0.556} \approx 0.746
$$

В датасете California Housing целевая переменная MedHouseVal измеряется в сотнях тысяч долларов.

Поэтому:

$$
0.746 \cdot 100000 \approx 74600
$$

То есть типичная ошибка модели по масштабу составляет примерно 75 000 долларов.

Это довольно заметная ошибка, поэтому модель нельзя считать очень точной.

Полученное значение:

$$
R^2 = 0.576
$$

означает, что модель объясняет примерно 57.6% вариации стоимости жилья.

Итог: линейная регрессия улавливает общую зависимость между признаками и стоимостью жилья, но предсказывает цену не очень точно.

## Задание 5
Смотрим веса модели для каждого признака и определяем самый влиятельный признак по наибольшему модулю веса.


In [5]:
weights = pd.DataFrame({
    "Признак": X.columns,
    "Вес": model.coef_
})

print(weights)

# Ищем самый большой вес по модулю
most_important = weights.iloc[weights["Вес"].abs().argmax()]

print("\nСамый влиятельный признак:", most_important["Признак"])
print("Его вес:", most_important["Вес"])


      Признак       Вес
0      MedInc  0.448675
1    HouseAge  0.009724
2    AveRooms -0.123323
3   AveBedrms  0.783145
4  Population -0.000002
5    AveOccup -0.003526
6    Latitude -0.419792
7   Longitude -0.433708

Самый влиятельный признак: AveBedrms
Его вес: 0.7831449067929722


## Дополнительное задание
Стандартизируем признаки с помощью StandardScaler, заново обучаем модель и сравниваем её веса и качество с исходной моделью.


### Стандартизация признаков

Стандартизация приводит признаки к сопоставимому масштабу.

Формула стандартизации:

$$
z = \frac{x - \mu}{\sigma}
$$

где:

- x — исходное значение;
- μ — среднее значение признака;
- σ — стандартное отклонение;
- z — стандартизированное значение.

Стандартное отклонение:

$$
\sigma = \sqrt{\frac{1}{N}\sum_{i=1}^{N}(x_i-\mu)^2}
$$

После стандартизации среднее значение признака становится примерно равно 0, а стандартное отклонение — примерно 1.

Стандартизация нужна, чтобы признаки с разными масштабами было удобнее сравнивать между собой.

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Стандартизируем признаки
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Обучаем новую модель
model_scaled = LinearRegression()
model_scaled.fit(X_train_scaled, y_train)

# Делаем прогноз
y_pred_scaled = model_scaled.predict(X_test_scaled)

# Считаем метрики
mse_scaled = mean_squared_error(y_test, y_pred_scaled)
r2_scaled = r2_score(y_test, y_pred_scaled)

print("MSE до стандартизации:", mse)
print("MSE после стандартизации:", mse_scaled)
print("R² до стандартизации:", r2)
print("R² после стандартизации:", r2_scaled)

# Смотрим новые веса
weights_scaled = pd.DataFrame({
    "Признак": X.columns,
    "Вес после стандартизации": model_scaled.coef_
})

print("\nВеса после стандартизации:")
print(weights_scaled)


MSE до стандартизации: 0.5558915986952444
MSE после стандартизации: 0.5558915986952442
R² до стандартизации: 0.5757877060324508
R² после стандартизации: 0.575787706032451

Веса после стандартизации:
      Признак  Вес после стандартизации
0      MedInc                  0.854383
1    HouseAge                  0.122546
2    AveRooms                 -0.294410
3   AveBedrms                  0.339259
4  Population                 -0.002308
5    AveOccup                 -0.040829
6    Latitude                 -0.896929
7   Longitude                 -0.869842


### Краткий вывод по дополнительному заданию
После стандартизации веса модели изменяются, потому что признаки теперь имеют одинаковый масштаб.  
Качество модели (MSE и R²) должно остаться практически таким же.
